<a href="https://www.kaggle.com/code/ruhikaragl/improving-dementia-diagnosis-with-synthetic-brain?scriptVersionId=246157932" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.inception_v3 import InceptionV3
import shutil
from sklearn.model_selection import train_test_split
import os
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, BatchNormalization, Dropout, Flatten, Dense, Activation, MaxPool2D, Conv2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.callbacks import ReduceLROnPlateau

from tensorflow.keras.utils import to_categorical
from PIL import Image
from keras.applications.vgg16 import VGG16
import matplotlib.pyplot as plt
from glob import glob
import numpy as np
import tensorflow as tf
import tensorflow

from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization,GlobalAveragePooling2D

from distutils.dir_util import copy_tree, remove_tree

**LOOKING DATA**

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image  # PIL modülü, görüntü işlemleri için

# Define paths to your directories
orijinal = "/kaggle/input/alzheimerbasic/alzkarısık/train/MildDemented"
sentetik = "/kaggle/input/amnakoyim-yeter/cycgan/mod"

# Get list of files
orijinal_images = [os.path.join(orijinal, f) for f in os.listdir(orijinal)]
sentetik_images = [os.path.join(sentetik, f) for f in os.listdir(sentetik)]

# Randomly select 3 from each class
selected_orijinal = random.sample(orijinal_images, 3)
selected_sentetik = random.sample(sentetik_images, 3)

# Combine and shuffle
combined = [(img, "Orijinal Demans") for img in selected_orijinal] + \
           [(img, "Sentetik CycleGAN Demans ") for img in selected_sentetik]
random.shuffle(combined)

# Plot results
plt.figure(figsize=(15, 10))
for idx, (img_path, label) in enumerate(combined):
    plt.subplot(2, 3, idx+1)
    # Open, grayscale'e çevir ve 128x128 boyutuna yeniden boyutlandır
    img = Image.open(img_path).convert("L")
    img = img.resize((128, 128))
    plt.imshow(img, cmap="gray")
    plt.title(label, fontsize=20)
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
import os
import shutil
import random

# Kaynak ve hedef dizinler
source_train_dir = '/kaggle/input/alzheimerbasic/alzkarısık/train'
destination_train_dir = '/kaggle/working/output/train'

mild_source_dir = '/kaggle/input/amnakoyim-yeter/cycgan/mild'
mod_source_dir = '/kaggle/input/amnakoyim-yeter/cycgan/mod'

# Her sınıf için hedef dizinleri oluştur
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

for cls in classes:
    os.makedirs(os.path.join(destination_train_dir, cls), exist_ok=True)

# 1. Adım: Tüm sınıfları birleştir ve hedefe taşı
for cls in classes:
    source_class_dir = os.path.join(source_train_dir, cls)
    destination_class_dir = os.path.join(destination_train_dir, cls)

    if os.path.exists(source_class_dir):
        for file_name in os.listdir(source_class_dir):
            source_file = os.path.join(source_class_dir, file_name)
            destination_file = os.path.join(destination_class_dir, file_name)
            shutil.copy(source_file, destination_file)

# 2. Adım: Ek veri aktarma (MildDemented ve ModerateDemented)

num_mild_files_to_add = 343  # Bu değeri ihtiyacınıza göre değiştirebilirsiniz
num_mod_files_to_add = 123  # Bu değeri ihtiyacınıza göre değiştirebilirsiniz

def transfer_files(source_dir, destination_dir, num_files):
    if os.path.exists(source_dir):
        files = os.listdir(source_dir)
        random.shuffle(files)
        files_to_transfer = files[:num_files]

        for file_name in files_to_transfer:
            source_file = os.path.join(source_dir, file_name)
            destination_file = os.path.join(destination_dir, file_name)
            shutil.copy(source_file, destination_file)

# MildDemented'e dosya ekle
transfer_files(mild_source_dir, os.path.join(destination_train_dir, 'MildDemented'), num_mild_files_to_add)

# ModerateDemented'e dosya ekle
transfer_files(mod_source_dir, os.path.join(destination_train_dir, 'ModerateDemented'), num_mod_files_to_add)

print("Dosyalar başarıyla taşındı ve birleştirildi.")

In [ ]:
import shutil
import random

def split_data(train_dir, output_dir, test_ratio=0.2):
    # Test veri klasörünü oluştur
    test_dir = os.path.join(output_dir, "test")
    train_output_dir = os.path.join(output_dir, "train")
    os.makedirs(test_dir, exist_ok=True)
    os.makedirs(train_output_dir, exist_ok=True)
    
    # Her sınıf için işlem yap
    for class_name in os.listdir(train_dir):
        class_path = os.path.join(train_dir, class_name)
        
        if not os.path.isdir(class_path):
            continue
        
        # Sınıfın test ve train klasörlerini oluştur
        test_class_path = os.path.join(test_dir, class_name)
        train_class_path = os.path.join(train_output_dir, class_name)
        os.makedirs(test_class_path, exist_ok=True)
        os.makedirs(train_class_path, exist_ok=True)
        
        # Sınıfın tüm dosyalarını al
        files = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
        
        # Test için seçilecek dosya sayısını belirle
        num_test_samples = int(len(files) * test_ratio)
        test_files = random.sample(files, num_test_samples)
        
        # Test verilerini kopyala
        for file_name in test_files:
            src_file = os.path.join(class_path, file_name)
            dest_file = os.path.join(test_class_path, file_name)
            shutil.copy(src_file, dest_file)
        
        # Train verilerini kopyala
        train_files = [f for f in files if f not in test_files]
        for file_name in train_files:
            src_file = os.path.join(class_path, file_name)
            dest_file = os.path.join(train_class_path, file_name)
            shutil.copy(src_file, dest_file)
    
    print(f"Test ve train verileri {output_dir} klasörüne kopyalandı.")

# Ana kod
if __name__ == "__main__":
    # Train veri yolu
    train_dir = "/kaggle/working/output/train"
    
    # Çıkış klasörü yolu
    output_dir = "/kaggle/working/alz"
    
    # Verileri ayır ve kopyala
    split_data(train_dir, output_dir, test_ratio=0.2)


In [ ]:
train_path='/kaggle/working/alz/train'
test_path='/kaggle/working/alz/test'

In [ ]:
invest=tf.keras.utils.image_dataset_from_directory(train_path)
class_names = invest.class_names
print(class_names)
plt.figure(figsize=(10, 10))
for images, labels in invest.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
numberOfClass = (glob(train_path+'/*'))
numberOfClass

In [ ]:
img_width=128
img_height=128
BATCH_SIZE=32
EPOCHS=200
SEED= 142


In [ ]:
# import os
# from tensorflow.keras.preprocessing.image import img_to_array, load_img
# import numpy as np
# from tensorflow.keras.preprocessing.image import ImageDataGenerator

# train_dir = '/kaggle/working/train_new'  # Eğitim veri setinizin yolu

# # Yalnızca 'ModerateDemented' ve 'MildDemented' için augmentation
# augmented_gen = ImageDataGenerator(
#     rescale=1./255,
#     rotation_range=20,  # Döndürme
#     width_shift_range=0.2,  # Yatay kaydırma
#     height_shift_range=0.2,  # Dikey kaydırma
#     zoom_range=0.2,  # Yakınlaştırma
#     horizontal_flip=True,  # Yatay çevirme
# )

# # Dosya yollarında her bir sınıfın alt klasörleri olduğunu varsayıyoruz
# # Bu şekilde sadece belirli klasörlere augmentation uygulayabiliriz
# for class_name in ['MildDemented', 'ModerateDemented']:
#     class_dir = os.path.join(train_dir, class_name)
    
#     # Her dosya için augmentasyon uygulayalım
#     for filename in os.listdir(class_dir):
#         img_path = os.path.join(class_dir, filename)
        
#         # Resmi yükleyip augmentasyon uygulayalım
#         img = load_img(img_path)  # Resmi yükle
#         img_array = img_to_array(img)  # NumPy dizisine çevir
#         img_array = np.expand_dims(img_array, axis=0)  # Batch boyutunu ekle
        
#         # Augmentasyon uygulama
#         aug_iter = augmented_gen.flow(img_array, batch_size=1, 
#                                       save_to_dir=class_dir,  # Augmentasyonlu görselleri aynı klasöre kaydet
#                                       save_prefix='aug_',  # Yeni dosya isimleri için ön ek
#                                       save_format='jpg')

#         # Augmentasyonlu resimleri kaydedelim
#         for _ in range(1):  # 1 kez augmentasyon yapacağız
#             next(aug_iter)  # augmentasyon örneğini al

# print("Augmentasyon başarıyla tamamlandı.")


In [ ]:
# Verileri hazırlayın


In [ ]:
train_gen = ImageDataGenerator( validation_split=0,
                                rescale = 1./255
                                
                              )
train_data = train_gen.flow_from_directory(
                                directory=train_path,
                                class_mode='categorical',
                                subset='training',
                                shuffle=True,
                                color_mode='rgb',
                                seed=SEED,
                                batch_size=BATCH_SIZE,
                                target_size=(img_height, img_width)
                        )

ts_gen = ImageDataGenerator(rescale=1.0 / 255)
test_data = ts_gen.flow_from_directory(
                                directory=test_path,
                                class_mode='categorical',
                                color_mode='rgb',
                                batch_size=BATCH_SIZE,
                                shuffle=False,
                                target_size=(img_height, img_width)
                        )


image_generator_submission = ImageDataGenerator(rescale=1/255,validation_split=0.2)
valid_data =image_generator_submission.flow_from_directory(
                                directory=train_path,
                                class_mode='categorical',
                                subset='validation',
                                color_mode='rgb',
                                seed=SEED,
                                shuffle=True,
                                batch_size=BATCH_SIZE,
                                target_size=(img_height, img_width)
                        )

In [ ]:
from keras import backend as K
K.clear_session()

In [ ]:
from tensorflow.keras.applications import ConvNeXtTiny
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.regularizers import l2

# ConvNeXtTiny modelini yükleyin
convnext_model =tf.keras.applications.ConvNeXtBase (weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))

# Katmanları dondurun
for layer in convnext_model.layers:
    layer.trainable = False

# Yeni katmanlar ekleyin
x = convnext_model.output
x = Flatten()(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu', kernel_regularizer=l2(0.01))(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
x = BatchNormalization()(x)

# Çıkış katmanı
predictions = Dense(4, activation='softmax')(x)

# Modeli oluşturun
model = Model(inputs=convnext_model.input, outputs=predictions)

# Model özeti
model.summary()

In [ ]:
# Model özeti
model.summary()

In [ ]:
class MyCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if logs.get('auc') > 0.99:
            print("\nReached accuracy threshold! Terminating training.")
            self.model.stop_training = True
            
my_callback = MyCallback()
early_stopping = EarlyStopping(monitor='accuracy', patience=5, restore_best_weights=True)
#ReduceLROnPlateau to stabilize the training process of the model
rop_callback = ReduceLROnPlateau(monitor="val_loss", patience=3)
callback= [early_stopping]

In [ ]:
from tensorflow.keras.optimizers import Adam


metrics = [
    tf.keras.metrics.AUC(name='auc'),
    tf.keras.metrics.Accuracy(name='acc')
]
model.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),  # Lower initial learning rate
    metrics=["accuracy", "AUC"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping



hist = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=EPOCHS,
    verbose = 1,
    callbacks=callback
)

In [ ]:
model.save("model1.h5")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(6, 6))  # Grafik boyutu küçültüldü
plt.subplots_adjust(hspace=0.5)  # Grafikler arası dikey boşluk

for i, met in enumerate(['accuracy', 'loss']):
    ax[i].plot(hist.history[met])
    ax[i].plot(hist.history['val_' + met])
    ax[i].set_title('ConvNeXtBase modeli {}'.format('doğruluk' if met == 'accuracy' else 'kayıp'))
    ax[i].set_xlabel('iterasyon')
    ax[i].set_ylabel('doğruluk' if met == 'accuracy' else 'kayıp')
    ax[i].legend(['eğitim', 'doğrulama'])

plt.show()


In [ ]:
# ... (model tanımlama, derleme ve eğitme adımları)

print("\nEğitim ve Doğrulama Sonuçları:")
print("-----------------------------")
print(f"Son Eğitim Doğruluğu (Accuracy): %{hist.history['accuracy'][-1]*100:.2f}") 
print(f"Son Doğrulama Doğruluğu (Accuracy): %{hist.history['val_accuracy'][-1]*100:.2f}") 
print(f"Son Eğitim Kaybı (Loss): {hist.history['loss'][-1]:.2f}")
print(f"Son Doğrulama Kaybı (Loss): {hist.history['val_loss'][-1]:.2f}") 


In [ ]:
# ... (model tanımlama, derleme ve eğitme adımları)

results = model.evaluate(test_data)

print("\nTest Verisi Üzerindeki Sonuçlar:")
print("---------------------------------")
print(f"Ortalama Kayıp (Loss): {results[0]:.4f}")  # 4 ondalık basamak
print(f"Doğruluk (Accuracy): %{results[1]*100:.2f}")  # Yüzde olarak, 2 ondalık basamak


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Önce modelinizle tahminler yapın
y_pred = model.predict(test_data)

# Tahminlerin sınıf etiketlerini alın
y_pred_classes = np.argmax(y_pred, axis=1)

# Gerçek etiketleri alın
y_true = test_data.classes

# Sınıf etiketlerini ve sınıf isimlerini alın (örneğin, 0: 'Class_0', 1: 'Class_1' gibi)
class_labels = list(test_data.class_indices.keys())

# Karmaşıklık matrisini hesaplayın
cm = confusion_matrix(y_true, y_pred_classes)

# Doğruluk değerini hesaplayın
accuracy = accuracy_score(y_true, y_pred_classes)

# Sınıf isimlerini kullanarak karmaşıklık matrisini çizdirin
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("Karışıklık  Matrisi")
plt.show()

# Sınıf bazında doğruluk sonuçlarını gösteren bir rapor alın
print("Sınıf bazında Doğruluk Raporu:\n", classification_report(y_true, y_pred_classes, target_names=class_labels))

# Genel doğruluk sonucunu görüntüleyin
print(f"Genel Doğruluk: %{accuracy * 100:.2f}")  